## Setup and Imports
This cell imports necessary libraries for PyTorch, neural networks, and tokenizer operations, along with `time` for performance measurement.

In [1]:
from tokenizers import Tokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

## Device Configuration
This cell detects and sets the device (GPU if available, otherwise CPU) for PyTorch operations to leverage hardware acceleration.

In [2]:
#device

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device is { device}")

device is cpu


## Global Constants and Paths
This cell defines important constants such as paths to the tokenizer and dataset files, checkpoint path format, and hyperparameters for the model like `block_size`, `batch_size`, `n_embd`, `n_head`, and `steps`.

In [3]:
#constants
tokenizer_path =r"./tokenization/dialogtokens.json"
ds_path = r"./dataset/dailyDialogCleand.txt"
checkPointPath = "./checkpoints/CLLM-I{id}-S{step}.pth"

block_size = 128
batch_size = 64
n_embd = 96
n_head = 12

steps = 5000
head_size = n_embd // n_head

## Data Loading and Tokenization
This cell loads the pre-trained tokenizer, reads the text dataset, encodes it into numerical tokens, and converts it into a PyTorch tensor. It also determines the vocabulary size.

In [4]:
tokenizer = Tokenizer.from_file(tokenizer_path)
text = open(ds_path,encoding="utf-8",mode="r").read()
tokens = tokenizer.encode(text)

data = torch.tensor(
    tokens.ids,
    dtype=torch.long,
    device=device
)

vocab_size = tokenizer.get_vocab_size()

## Train-Test Split and Batching Function
This cell splits the tokenized data into training and testing sets. It also defines a utility function `get_batch` to efficiently retrieve batches of input (`x`) and target (`y`) sequences for training or evaluation.

In [5]:
#train_test split
n = int(len(data) * 0.8)
train_data = data[:n]
test_data = data[n:]

def get_batch(split="train"):

    data = train_data if split == "train" else test_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

## Loss Estimation Function
This function, decorated with `@torch.no_grad()`, calculates the average loss on both the training and test datasets. It puts the model in evaluation mode to prevent gradient calculations during loss estimation and then switches it back to training mode.

In [6]:
@torch.no_grad()
def estimate_loss():
    out = {}
    losses = torch.zeros(50)
    model.eval()
    for split in ["train", "test"]:

        for i in range(50):

            Xb, Yb = get_batch(split)

            logits = model(Xb)
            loss = F.cross_entropy(logits.transpose(1, 2),Yb)

            losses[i] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out


## Model Architecture: Transformer Blocks
This cell defines the core components of a Transformer-based language model, including `FeedForward` networks, `Head` for single-attention, `MultiHeadAttention` for parallel attention, and `TransformerBlock` combining attention and feed-forward layers. Finally, `LanguageModel` puts these blocks together with token and positional embeddings.

In [7]:
class FeedForward(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.net = nn.Sequential(
            nn.Linear(n_embd,4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 *n_embd,n_embd)
        )
    def forward(self,x):
        return self.net(x)

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.key = nn.Linear(n_embd,head_size,bias=False)
        self.query = nn.Linear(n_embd,head_size,bias=False)
        self.value = nn.Linear(n_embd,head_size,bias=False)
        self.register_buffer(
                    "tril",
                    torch.tril(
                        torch.ones(block_size, block_size)
                    )
                )
    def forward(self,x):
        B,T,C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1)
        wei = wei * (k.shape[-1] ** -0.5)
        #Causal mask
        wei = wei.masked_fill(
            self.tril[:T,:T] == 0,
            float("-inf")
        )

        wei = F.softmax(
            wei,
            dim = -1
        )

        out = wei @ v

        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.heads = nn.ModuleList([
                Head()
                for _ in range(n_head)
            ])
        self.proj = nn.Linear(n_embd,n_embd)
    def forward(self, x):
        x = torch.cat(
                [head(x) for head in self.heads],
                dim=-1
            )

        return self.proj(x)



class TransformerBlock(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.attention = MultiHeadAttention()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ffw = FeedForward()
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffw(self.ln2(x))

        return x

class LanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embbeding = nn.Embedding(vocab_size,n_embd,device=device) #(B,T,C)
        self.pos_embedding = nn.Embedding(block_size,n_embd,device=device)
        self.blocks = nn.Sequential(
                    *[
                        TransformerBlock()
                        for _ in range(4)
                    ]
                )
        self.out = nn.Linear(n_embd ,vocab_size)

    def forward(self,x):
        B,T = x.shape
        t_embd = self.token_embbeding(x)
        p_embd = self.pos_embedding(torch.arange(T,device=device))
        x = t_embd + p_embd
        x = self.blocks(x)
        logits = self.out(x)
        return logits


## Model and Optimizer Initialization
This cell initializes the `LanguageModel` and moves it to the specified `device` (GPU/CPU). It also sets up the AdamW optimizer with a learning rate of `1e-3`.

In [8]:
model = LanguageModel()
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)

## Model Parameter Count
This cell calculates and prints the total number of trainable parameters in the model. This is an important metric to understand the model's complexity.Total learnable parameters is 5,754,802

In [29]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total learnable parameters: {total_params:,}")

Total learnable parameters: 5,754,802


## Training Loop
This cell implements the main training loop. It iterates for a predefined number of `steps`, retrieves batches, computes loss, performs backpropagation, and updates model parameters. It also includes logic for periodic loss estimation, checkpoint saving, and an early stopping mechanism to prevent overfitting.

In [ ]:


times = torch.zeros(steps)

test_losses = []
overfitting_counter =0
for s in range(steps):
    start = time.time()
    Xb,Yb = get_batch()
    logits = model(Xb)
    loss = F.cross_entropy(logits.transpose(1, 2),Yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    end = time.time()
    total = end - start
    times[s] = total
    if s % 10 == 0:
        mean_time = times[:s].mean()
        estimated = mean_time * (steps - s)
        e_l= estimate_loss()
        print(f"{s} - train loss: {e_l["train"].item()} test loss: {e_l["test"].item()} ---estimated time:({estimated})")
        test_losses.append(e_l["test"])

        if len(test_losses) >= 2 and test_losses[s//10] > test_losses[(s//10) - 1]:
            overfitting_counter += 1
        if overfitting_counter >= 3:
            print("overfitting break!")
            break
        else:
            overfitting_counter = 0
    if s % 100 == 0:
        torch.save({
            "model_state_dict":model.state_dict(),
            "optimizer_state_dict":optimizer.state_dict(),
            "loss":loss,
            "step":s
        },checkPointPath.format(id=0,step=s))



torch.save(model.state_dict(),"./model1.pth")

## Checkpoint Loading for Testing
This cell handles the loading of a previously saved model checkpoint. It prompts the user for a checkpoint ID and step, then loads the model and optimizer states, and reports the current step and loss from the checkpoint.

In [11]:
# test the model
# load check point:
load_checkpoint = input("what is the checkpoint id?\n")
load_checkpoint = int(load_checkpoint) if load_checkpoint else None
checkpoint_step = input("what is the checkpoint step?\n")
checkpoint_step = int(checkpoint_step) if checkpoint_step else None
if checkpoint_step:
    checkpoint = torch.load(checkPointPath.format(id=load_checkpoint,step=checkpoint_step),map_location = torch.device(device))
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    loss = checkpoint["loss"]
    start_step = checkpoint["step"]
    print(f"checkpoint load successfully currnet step is{start_step}, current loss is {loss}")
else:
    print("no checkpoint found")
    start_step = 0

checkpoint load successfully currnet step is4900, current loss is 2.8093161582946777


## Test Dataset Preparation
This cell loads the separate test dataset, encodes it using the tokenizer, and converts it into a PyTorch tensor, similar to how the training data was processed.

In [12]:
#test dataset

test_text = open(r"./dataset/dailyDialogCleandTest.txt",encoding="utf-8",mode="r").read()
test_tokens = tokenizer.encode(test_text)
test_data = torch.tensor(
    test_tokens.ids,
    dtype=torch.long,
    device=device)


## Prepare Test Batches
This cell sets the model to evaluation mode and generates random indices to create `Xtest` (input sequences) and `Ytest` (target sequences) from the `test_data` for model evaluation.

In [13]:
model.eval()
ix = torch.randint(
        len(test_data) - block_size,
        (64,)
    )
Xtest = torch.stack([
        test_data[i:i + block_size]
        for i in ix]).to(device)

Ytest = torch.stack([
        test_data[i + 1:i + block_size + 1]
        for i in ix]).to(device)


## Calculate Test Loss
This cell computes the cross-entropy loss for 100 batches of the test data (`Xtest`, `Ytest`) using the loaded model and then prints the average loss, providing an overall measure of the model's performance on unseen data.

In [14]:
all_losses = torch.zeros(100)
for i in range(100):
    logits = model(Xtest)
    loss = F.cross_entropy(logits.transpose(1, 2),Ytest)
    all_losses[i] = loss.item()

print(all_losses.mean())

tensor(4.1979)


# Using the Model

In [15]:
def generate(prompt:str):

    model.eval()
    prompt = "<user> " + prompt + "<assistant> "

    idx = torch.tensor(
        tokenizer.encode(prompt).ids,
        dtype=torch.long
    ).unsqueeze(0)


    with torch.no_grad():
        for _ in range(300):

            idx_cond = idx[:, -block_size:]
            logits = model(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )
            idx = torch.cat(
                (idx, next_token),
                dim=1
            )
            if next_token in (1,2):
                break
    result = tokenizer.decode(idx[0].tolist(),skip_special_tokens=False)

    print(result)


In [16]:
while True:
    prompt = input(">>> ")
    if prompt == "exit":
        break
    else:
        generate(prompt)

<user> hello <assistant> I have to make some clothes to know . You are your parents . <user>
<user> what is your name ? <assistant> I beg it from Nick with my office , yes , and I have lost your phone number — now Account , I would be able to check it in a department . <user>
